# E06 — AGU-S: projeto completo, parte 1

Este estudo reconstrói a rede a partir de um acesso ao backbone e 32 rádios com equipamentos associados, usando 22.078 postes como candidatos opcionais. Os enlaces e repetidores anteriores não são preservados. Requer SDK 0.1.78 ou superior e os serviços de rede e features atualizados.

Os 27 rádios com DA são folhas; os cinco rádios somente AMI podem retransmitir. O inventário contém 39 equipamentos, com carga simulada individual em `attached_equipment.csv`: DA = 1 unidade de tráfego; AMI = 5–20. Nesta parte esses valores não influenciam o planejamento. Limites de caminho e de carga por ramo pertencem à parte 2.

O modelo de RF e os ganhos vêm do estudo AGU-S/E08. As antenas YAGI mantêm o ganho informado, mas não têm orientação nem diagrama disponíveis. Consulte `README.md` para as premissas, o recorte do inventário e os parâmetros de busca.

Depois da etapa 1, você pode retomar na etapa 2 a partir do checkpoint, começar na etapa 4 se a expansão já estiver concluída, ou ir direto à etapa 5 se a solução final já estiver salva.

## 1. Definir projeto e métrica da tecnologia

Abra o projeto AGU-S com o planejador mesh e carregue a métrica da tecnologia GE.

O arquivo `metrics/e06_ge_rssi_rank.toml` conserva o estimador de RSSI treinado na E08. O RSSI previsto seleciona a equação. Com sinal igual ou melhor que o limite de −80 dBm, o custo vale 1 no ideal (−65 dBm ou melhor) e cresce quadraticamente até 3 no limite. Com sinal pior que o limite, a métrica soma um salto adicional de custo 3 e uma parcela quadrática que cresce com a piora do sinal.

O rank soma os custos dos enlaces. No cenário distribuído, o limite inicial por segmento é 3,0 e o cálculo considera o pior sentido. Resultados salvos mantêm os custos da métrica usada na execução.

In [ ]:
from cisei_planning_sdk import PlanningClient, PlanningProject

client = PlanningClient("http://network-service:8080", workspace="..", timeout=14400)
project = PlanningProject("../projects/e06_agu_redesign")
scenario = client.open_project(project, planner="mesh")
configuration = scenario.upload_configuration(metrics={"ge_mesh": "e06_ge_rssi_rank.toml"})

## 2. Adicionar repetidores

Cada execução estuda uma interface pendente, salva o checkpoint e abre o visualizador na última iteração disponível. O visualizador é recriado quando a chamada termina. Enquanto a rodada roda, a caixa de progresso acompanha a execução.

Se o estado for **Paused**, repita a célula para continuar. A chamada retoma o último resultado salvo. Use `resume=False` apenas para iniciar uma nova expansão. Quando a expansão terminar ou ficar sem novas tentativas, passe para a etapa 3 ou, se os caminhos já estiverem adequados, para a etapa 4.

O visualizador abre na última iteração. Use a lista para voltar aos estados anteriores, incluindo o estado inicial 0000. Os símbolos distinguem acessos ao backbone, repetidores e terminais. Apenas os enlaces selecionados aparecem, com suas métricas e os ranks acumulados dos nós. Selecione um nó para destacar o caminho até o backbone e consultar o resumo. Nenhum planejamento é executado ao trocar a iteração.

Os argumentos passados na chamada têm precedência sobre os valores de `scenario.toml`.

| Argumento | Valor neste exemplo | Função |
|---|---|---|
| `geo_base_url` | `"http://features:8080"` | Endereço do serviço de extração de features. |
| `max_rounds` | `1` | Número máximo de rádios estudados por chamada. Uma rodada pode testar vários pais. Com `None`, a chamada roda até terminar. |
| `progress` | `True` | Mostra o andamento no notebook. O padrão do SDK é `False`. |
| `resume` | `True` | Retoma o checkpoint quando `max_rounds` está definido. Com `False`, inicia uma nova expansão. |
| `poles` | `None` | Por padrão, carrega `poles.csv` conforme `[mesh.poles]`. Aceita uma lista de postes fornecida pelo usuário. |
| `parent_candidates` | `2` | Número máximo de pais conectados e compatíveis testados por rádio. |
| `corridor_m` | `1500` | Distância lateral máxima, em metros, de um poste ao segmento direto entre o rádio e o pai. |
| `max_candidates` | `5` | Limite de postes candidatos a cada passo da busca. |
| `max_repeaters` | `2` | Limite de novos repetidores no caminho até o pai. Com `0`, testa a conexão direta. |
| `segment_threshold` | `3.0` | Custo máximo aceitável por segmento, considerando a pior direção. |
| `search_heuristic` | `"sector_band"` | Estratégia de busca de postes. A outra opção é `"dual_cone"`. |
| `search_midpoint` | `True` | Inclui a região do ponto médio na busca. |
| `assume_symmetric_features` | `True` | Reutiliza a extração de features nas duas direções e calcula o custo de cada direção. |
| `final_degree` | `3` | Grau das regras RF padrão na montagem posterior do grafo candidato. |
| `retry_unserved` | `False` | Com `True`, usa novos parâmetros para retomar a expansão dos rádios ainda desconectados. Veja a célula da segunda busca. |

In [ ]:
construction = scenario.add_repeaters(geo_base_url="http://features:8080", progress=True, max_rounds=1)
construction_network = scenario.show_mesh_iterations()
construction_network

## 3. Melhorar caminhos da rede

Você pode estudar caminhos manualmente no **Improve path** ou fazer uma nova busca automática para os rádios ainda desconectados.

**Abordagem manual.** Depois de uma rodada de expansão, use **Improve path** para estudar uma alternativa para um rádio sem conexão ou cujo caminho atual tenha um segmento de baixa qualidade. Selecione o rádio e um pai conectado. Para procurar uma alternativa entre os mesmos pontos de um segmento existente, selecione o rádio a jusante e mantenha seu pai atual. Você pode ampliar o corredor de busca, aumentar o número de postes candidatos ou ajustar o limite de repetidores para esse estudo.

**Try path** registra a tentativa e mostra o caminho proposto, seus repetidores, os custos dos segmentos e o rank acumulado. Compare esses valores com os do caminho atual, quando houver. Se a proposta atender ao limite de custo por segmento, **Apply proposal** incorpora o caminho escolhido em uma nova iteração e preserva as conexões a jusante do rádio. Use **Recorded study** para consultar as tentativas registradas.

In [ ]:
scenario.show_mesh_study(geo_base_url="http://features:8080")

### Busca automática para rádios ainda desconectados

Depois da busca inicial, execute esta célula se ainda houver rádios sem conexão. O planejador retoma a última iteração, preserva os caminhos construídos e aplica os novos parâmetros aos estudos dos rádios pendentes. Neste exemplo, o corredor passa para 2.500 m, a busca pode testar três pais e o caminho pode usar até três repetidores. O limite de custo por segmento continua sendo o do cenário.

Repita esta mesma célula para continuar em lotes de até três rodadas. Com os mesmos parâmetros, o checkpoint preserva a lista de rádios já tentados para a topologia atual. Use o visualizador de iterações para inspecionar o resultado antes de montar o grafo candidato final.

In [ ]:
construction = scenario.add_repeaters(
    geo_base_url="http://features:8080",
    progress=True,
    max_rounds=3,
    retry_unserved=True,
    corridor_m=2500,
    parent_candidates=3,
    max_repeaters=3,
)

## 4. Projetar rede final

**Carregar a expansão e inspecionar o grafo candidato.** A primeira célula carrega a expansão concluída ou encerrada sem novas alternativas, monta o grafo e abre a visualização dos candidatos. Agora usamos o GraphPlanner sobre os rádios originais e os repetidores propostos. A operação constrói alternativas de conexão pelas regras de conectividade e não consulta features, não procura novos postes e não acrescenta equipamentos.

Use esta célula após reabrir o projeto para retomar o projeto final. Ela também permite voltar à rede expandida depois de uma poda anterior.

In [ ]:
expanded = scenario.load_mesh_expansion()
candidates = scenario.build_candidate_edges()
candidate_network = scenario.show_network(kind="candidate")

**Calcular as métricas, selecionar os caminhos e salvar.** A segunda célula percorre lotes de até dez enlaces, salva o cache de features após cada lote e usa o maior custo entre os dois sentidos. Em seguida, seleciona os caminhos com `max_edge_metric=10` e mantém até três pais por rádio, incluindo o preferencial. A poda retira os repetidores sem uso nos caminhos mantidos. Depois da poda, `evaluate_paths()` usa `max_load_weighted_path` para pontuar cada caminho pelo maior produto entre carga e custo de um enlace. Use `default_path` para somar apenas os custos. Em seguida, `save_result()` grava a solução e as notas em `result.json`.

Os estados da expansão ficam em `mesh_iterations`, e o projeto final fica em `mesh_solution`. O `scenario.toml` de saída contém as instâncias completas. O `nodes.csv` correspondente acompanha o inventário por site quando os perfis permitem essa representação.

Depois de avaliar os caminhos, `network_metric()` calcula a nota global. A métrica `default_network` usa a média dos caminhos dos rádios com equipamentos associados. A nota e o resumo são salvos junto com a solução.

In [ ]:
metrics = scenario.compute_metrics(geo_base_url="http://features:8080", batch_size=10)
solution = scenario.solve(parent_set_size=3, max_edge_metric=10)
pruning = scenario.remove_unused_repeaters()
path_metric = "max_load_weighted_path"
path_scores = scenario.evaluate_paths(metric=path_metric)
network_evaluation = scenario.network_metric(metric="default_network")
saved_result = scenario.save_result()

## 5. Inspecionar resultado final

Carregue o `result.json` já salvo e abra a visualização da solução. Você pode executar esta célula depois de abrir o projeto na etapa 1, pulando as etapas de planejamento. O painel mostra a nota salva do caminho preferencial, os pais alternativos do rádio selecionado e os dos rádios a montante, com a métrica de cada enlace alternativo. Os terminais sem caminho aparecem destacados.

In [ ]:
result = scenario.load_result()
solution_network = scenario.show_mesh_network()
solution_network

### Nota final e resumo da rede

A consulta usa a avaliação gravada em `result.json`. O resumo mostra a média, o pior caminho e os percentis P50, P90 e P95. Se a solução foi salva antes da inclusão da métrica de rede, execute a avaliação e salve novamente.

In [ ]:
network_evaluation = result.get("network_evaluation", {})
{key: network_evaluation[key] for key in ("network_score", "summary") if key in network_evaluation}